## 🎯 Learning Objectives
* Understand the scope and importance of the 'Building Advanced AI Agents with AutoGen' course (ADV-02) within the Agentic AI & Automation Tools track.
* Set up the development environment with AutoGen 0.4, including necessary dependencies and API configurations.
* Gain an initial understanding of AutoGen's core architecture, specifically the AgentChat and Core APIs.
* Execute a basic 'Hello World' agent interaction to confirm successful environment setup and grasp fundamental agent communication.


# Welcome to ADV-02: Building Advanced AI Agents with AutoGen

Welcome, advanced AI engineers and researchers, to ADV-02! This course is a pivotal step in your journey through the Agentic AI & Automation Tools track, building directly upon the foundational concepts established in AG-03. In 2026, the landscape of AI is increasingly defined by sophisticated, collaborative agent systems capable of autonomous problem-solving and complex task execution. AutoGen, with its flexible and powerful framework, stands at the forefront of this revolution.

### Why This Course Matters

The ability to design, implement, and deploy multi-agent systems is no longer a niche skill but a critical competency for anyone pushing the boundaries of AI. AutoGen 0.4, the version we'll be focusing on, has matured into a robust platform for orchestrating intricate agentic workflows. This course will equip you with the expertise to move beyond single-agent interactions, enabling you to build:

*   **Conversable Assistant Agents:** Agents that can engage in natural, multi-turn dialogues to achieve goals.
*   **Round-Robin/Selector Groups:** Dynamic agent teams where tasks are intelligently distributed and managed.
*   **Distributed Systems with AutoGen:** Architectures that leverage multiple agents across different environments for enhanced scalability and resilience.

### Course Overview: ADV-02

Throughout ADV-02, we will dive deep into AutoGen's advanced features, exploring patterns for complex agentic behaviors, robust error handling, and integration with external tools and services. You'll learn to design agents that not only communicate but truly collaborate, adapting their strategies in real-time to solve challenging problems.

### Lesson 1: Course Overview and AutoGen 0.4 Setup (AgentChat and Core APIs)

This introductory lesson sets the stage for our journey. We'll start by ensuring your development environment is perfectly configured for AutoGen 0.4. We'll then introduce the fundamental architectural components of AutoGen, focusing on the **AgentChat** and **Core APIs**, which are the bedrock of all agent interactions. By the end of this lesson, you'll have successfully run your first multi-agent 


## Prerequisites and Environment Setup

Before we dive into advanced agentic patterns, let's ensure our development environment is correctly configured. This course assumes you have successfully completed AG-03 and are familiar with basic AutoGen concepts, Python programming, and working with large language models (LLMs).

### Required Tools & Libraries (2026 Ready)

*   **Python 3.10+**: We recommend Python 3.11 or 3.12 for optimal performance and access to the latest language features.
*   **AutoGen 0.4.x**: We will specifically target AutoGen version 0.4.x, which introduces significant architectural improvements and API stability.
*   **Jupyter Notebook / Google Colab**: For an interactive coding experience.
*   **LLM API Key**: Access to an LLM provider (e.g., OpenAI, Azure OpenAI, Anthropic, Google Gemini) is essential. Ensure your API key is securely stored and accessible (e.g., via environment variables).

### Installation Steps

If you're using a local environment, open your terminal and run the following commands. If you're in Google Colab, you can run these directly in a code cell.

1.  **Create a virtual environment (recommended for local development):**
    ```bash
    python -m venv autogen_adv_env
    source autogen_adv_env/bin/activate  # On Windows: .\autogen_adv_env\Scripts\activate
    ```

2.  **Install AutoGen 0.4.x and its dependencies:**
    ```bash
    pip install "pyautogen~=0.4.0"
    ```
    *Note: The `~=0.4.0` ensures you get the latest patch version within the 0.4 series, which is crucial for stability and new features in 2026.*

3.  **Install additional dependencies (if needed, e.g., for specific LLM providers or tools):**
    For example, if using OpenAI:
    ```bash
    pip install openai
    ```

### Google Colab Specifics

If you're running this notebook in Google Colab, you can execute the `pip install` commands directly in a code cell by prefixing them with an exclamation mark (`!`). Colab environments are ephemeral, so you'll need to run the installation commands at the beginning of each session.

```python
# For Google Colab users, run this cell to install AutoGen
# !pip install "pyautogen~=0.4.0"
# !pip install openai # If using OpenAI models
```

### API Key Configuration

AutoGen requires access to an LLM. The most secure and recommended way to provide your API key is via environment variables. For example, for OpenAI, set `OPENAI_API_KEY`.

In a local environment, you can set it in your shell:

```bash
export OPENAI_API_KEY='YOUR_OPENAI_API_KEY'
```

In Google Colab, you can use `os.environ` or Colab's Secrets feature:

```python
import os

# Option 1: Directly set (not recommended for sensitive keys in shared notebooks)
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Option 2: Use Colab's Secrets feature (recommended for Colab)
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Verify the key is set
if "OPENAI_API_KEY" in os.environ:
    print("OPENAI_API_KEY found in environment variables.")
else:
    print("WARNING: OPENAI_API_KEY not found. Please set it before proceeding.")
    print("You can also pass it directly in the config_list, but environment variables are preferred.")
```

With your environment set up, we're ready to initialize our first agents!


In [ ]:
import autogen
import os

# --- 1. Configure LLM Settings ---
# In 2026, it's common to use a list of configurations for flexibility
# and fallback mechanisms. We'll start with a single configuration.

# Ensure your API key is set as an environment variable (e.g., OPENAI_API_KEY)
# For demonstration, we'll assume OpenAI's API is being used.

config_list = [
    {
        "model": "gpt-4o-2024-05-13", # A powerful, current model in 2026
        "api_key": os.environ.get("OPENAI_API_KEY"),
    }
    # You could add more configurations here for fallback or different models:
    # {"model": "gpt-3.5-turbo", "api_key": os.environ.get("OPENAI_API_KEY")},
    # {"model": "claude-3-opus-20240229", "api_key": os.environ.get("ANTHROPIC_API_KEY")},
]

# Check if API key is available
if not config_list[0]["api_key"]:
    raise ValueError("OPENAI_API_KEY environment variable not set. Please set it before running this cell.")

print("LLM configuration loaded successfully.")

# --- 2. Initialize Agents (AgentChat API) ---
# AutoGen's AgentChat API provides the core components for agent interaction.
# We'll create two basic agents: a UserProxyAgent and an AssistantAgent.

# The UserProxyAgent represents a human user or a proxy for human input.
# It can execute code and ask for human input if needed.
user_proxy = autogen.UserProxyAgent(
    name="User_Proxy",
    human_input_mode="NEVER", # For this demo, agents won't ask for human input
    max_consecutive_auto_reply=10, # Allow up to 10 consecutive replies without human intervention
    is_termination_msg=lambda msg: "TERMINATE" in msg["content"].upper(), # Define how the conversation ends
    code_execution_config={"work_dir": "coding", "use_docker": False}, # Configure code execution
    llm_config={"config_list": config_list}, # Attach LLM config
)

# The AssistantAgent is an AI agent that can write code, solve tasks, and respond to queries.
assistant = autogen.AssistantAgent(
    name="Assistant",
    llm_config={"config_list": config_list},
)

print("User_Proxy and Assistant agents initialized.")

# --- 3. Start a Basic Conversation (Core API) ---
# The `initiate_chat` function from the Core API orchestrates the conversation
# between agents.

print("\n--- Starting a basic 'Hello World' agent conversation ---")

chat_result = user_proxy.initiate_chat(
    assistant,
    message="Hello, Assistant! Can you tell me a simple fact about AutoGen 0.4?",
    clear_history=True # Start with a clean slate for this chat
)

print("\n--- Conversation Ended ---")
print(f"Final message: {chat_result.last_message['content']}")
print("This confirms your AutoGen 0.4 setup is working correctly!")
